## Задание 1

Переведите содержимое файла `purchase_log.txt` в словарь `purchases` вида:

```
{'1840e0b9d4': 'Продукты', ...}
```

Пример вывода первых двух элементов:

```
1840e0b9d4 'Продукты'
4e4f90fcfb 'Электроника'
```

### Решение

In [4]:
import json
from pathlib import Path


def file_error(path, title):
    """Проверка: путь указан и файл существует."""
    if not path or not str(path).strip():
        return f"Не указан путь к {title}."
    file_path = Path(path)
    if not file_path.exists():
        return f"Файл не найден: {path}"
    if not file_path.is_file():
        return f"Это не файл: {path}"
    return None


def load_purchases(path):
    """Прочитать purchase_log.txt в память и собрать словарь user_id -> категория."""
    purchases = {}
    with open(path, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            user_id = record.get("user_id")
            category = record.get("category")
            # Первая строка файла — заголовок в виде JSON.
            if not user_id or user_id == "user_id":
                continue
            if not category or category == "category":
                continue
            purchases[user_id] = category
    return purchases


def print_purchases_preview(purchases, count=2):
    for i, (user_id, category) in enumerate(purchases.items()):
        if i >= count:
            break
        print(f"{user_id} '{category}'")

In [5]:
# @title Покупки
purchase_path = "purchase_log.txt"  # @param {type:"string"}
preview_count = 2  # @param {type:"integer"}

error = file_error(purchase_path, "purchase_log.txt")
if error:
    print(error)
    purchases = {}
else:
    purchases = load_purchases(purchase_path)
    print_purchases_preview(purchases, preview_count)

1840e0b9d4 'Продукты'
4e4f90fcfb 'Электроника'


## Задание 2

Для каждого `user_id` в файле `visit_log.csv` определите третий столбец с категорией покупки. Сам `visit_log.csv` не меняйте.

Запишите в `funnel.csv` визиты, в которых были покупки, с указанием категории.

`purchase_log.txt` помещается в память. `visit_log.csv` — нет, читайте его только построчно.

Пример первых строк `funnel.csv`:

```
user_id,source,category
1840e0b9d4,other,Продукты
4e4f90fcfb,context,Электроника
```

### Решение

In [6]:
def write_funnel(visit_path, funnel_path, purchases):
    """Построчно обойти визиты и записать в funnel.csv только тех, кто покупал."""
    with open(visit_path, encoding="utf-8") as visits, open(
        funnel_path, "w", encoding="utf-8", newline=""
    ) as funnel:
        visits.readline()
        funnel.write("user_id,source,category\n")
        for line in visits:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) < 2:
                continue
            user_id, source = parts[0], parts[1]
            category = purchases.get(user_id)
            if category is None:
                continue
            funnel.write(f"{user_id},{source},{category}\n")


def print_funnel_preview(path, count=3):
    with open(path, encoding="utf-8") as file:
        for i, line in enumerate(file):
            if i >= count:
                break
            print(line.rstrip())

In [7]:
# @title Визиты и воронка
visit_path = "visit_log.csv"  # @param {type:"string"}
funnel_path = "funnel.csv"  # @param {type:"string"}
preview_count = 3  # @param {type:"integer"}

if not purchases:
    print("Сначала выполните задание 1.")
else:
    error = file_error(visit_path, "visit_log.csv")
    if error:
        print(error)
    else:
        write_funnel(visit_path, funnel_path, purchases)
        print_funnel_preview(funnel_path, preview_count)

user_id,source,category
1840e0b9d4,other,Продукты
4e4f90fcfb,context,Электроника


## Тесты

In [8]:
assert file_error("purchase_log.txt", "purchase_log.txt") is None
assert file_error("no_such_file.txt", "x") is not None
assert file_error("", "x") is not None

assert purchases["1840e0b9d4"] == "Продукты"
assert purchases["4e4f90fcfb"] == "Электроника"
first_two = list(purchases.items())[:2]
assert first_two == [
    ("1840e0b9d4", "Продукты"),
    ("4e4f90fcfb", "Электроника"),
]

with open("visit_log.csv", encoding="utf-8") as file:
    visit_header = file.readline().strip()
assert visit_header == "user_id,source"

with open("funnel.csv", encoding="utf-8") as file:
    funnel_head = [file.readline().rstrip() for _ in range(3)]
assert funnel_head == [
    "user_id,source,category",
    "1840e0b9d4,other,Продукты",
    "4e4f90fcfb,context,Электроника",
]

print("Все тесты пройдены")

Все тесты пройдены
